# Notebook 2 of 2 — Build the Silver layer

**Your job:** read your Bronze tables, find out what is wrong with the data, and publish **trustworthy Silver** tables, with the bad rows kept in a rejected table so nothing disappears silently.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract

| Output | ADLS path (container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Clean trips | `silver/taxi_trips/` | `<catalog>.silver.taxi_trips` |
| Rejected trips | `silver/taxi_trips_rejected/` | `<catalog>.silver.taxi_trips_rejected` |
| Clean weather | `silver/weather_daily/` | `<catalog>.silver.weather_daily` |

External Delta tables again, same storage account and container as Bronze. Silver writes use `overwrite`, so the notebook can be re-run safely.

In [ ]:
# ---- Fill in the two values marked <...> (same values as notebook 1) ----------
STORAGE_ACCOUNT = "<your_storage_account>"
CATALOG         = "<your_catalog>"

# ---- Do NOT change these ------------------------------------------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

# The 20 columns exactly as the NYC TLC publishes them (Bronze keeps these names)
SRC_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
    "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
    "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
]

---
## Step 1 — Look before you clean (profile Bronze)

Write queries that answer these four questions. Put the numbers in report §5.

1. How many rows in total, and how many nulls per column?
2. What is the earliest and latest pickup? How many pickups fall **outside the month of the file they came from**?
3. What values does `payment_type` take, and how many rows each?
4. How many trips have dropoff ≤ pickup, distance ≤ 0, fare ≤ 0?

**Hint:** one pass with `F.sum(F.col(c).isNull().cast("int"))` over `SRC_COLS` gives all null counts at once.

**Pause and observe →** Several columns show the *exact same* huge null count. Which columns, and which `payment_type` value has that same count? What does that tell you about those rows? Would deleting them be right? Decide, and defend it in report §6.

In [ ]:
# Step 1: your code here

---
## Step 2 — Split off the rejects

A row is **rejected** if it breaks any rule below. Each rejected row gets exactly **one** `reject_reason` (the **first** rule it fails, in this order):

| # | `reject_reason` (exact string) | Rejected when |
|---|---|---|
| 1 | `pickup_outside_source_month` | the pickup month ≠ `source_month` |
| 2 | `dropoff_not_after_pickup` | dropoff ≤ pickup |
| 3 | `duration_over_6h` | trip lasts more than 360 minutes |
| 4 | `distance_invalid` | `trip_distance` ≤ 0 or ≥ 100 miles |
| 5 | `fare_invalid` | `fare_amount` ≤ 0 or ≥ 500 |

Write rejected rows to `silver/taxi_trips_rejected/`: **all Bronze columns unchanged**, plus `reject_reason` (string) and `rejected_at` (timestamp).

**Hints**
- A chain of `F.when(...).when(...)` returns the first match, which is exactly "first failing rule".
- Keep the original column names and types here. This table is evidence for someone investigating.
- Count rejected rows per reason and keep **one example row per reason** for the report.

**Pause and observe →** Why does the table keep the *original* values instead of cleaned ones? Why one reason per row? Report §6.

In [ ]:
# Step 2: your code here

---
## Step 3 — Clean the good rows into `silver.taxi_trips`

Take the rows that passed Step 2, **remove exact duplicates** (all 20 source columns equal; count how many you removed into a variable named `DUPLICATES_REMOVED`), and shape them exactly like this:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `VendorID` | `vendor_id` | int | |
| `tpep_pickup_datetime` | `pickup_ts` | timestamp | |
| `tpep_dropoff_datetime` | `dropoff_ts` | timestamp | |
| `passenger_count` | `passenger_count` | int | outside 1–6 → `NULL` (keep the trip, fix the value) |
| `trip_distance` | `trip_distance_miles` | decimal(8,2) | |
| `RatecodeID` | `rate_code_id` | int | |
| `store_and_fwd_flag` | `store_and_fwd_flag` | boolean | `Y` → true, `N` → false, anything else → null |
| `PULocationID` | `pickup_location_id` | int | |
| `DOLocationID` | `dropoff_location_id` | int | |
| `payment_type` | `payment_type_id` | int | |
| *(new)* | `payment_type_desc` | string | 0 Flex fare · 1 Credit card · 2 Cash · 3 No charge · 4 Dispute · 5 Unknown · 6 Voided trip · else `Other` |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `cbd_congestion_fee` | same names | decimal(10,2) | money is never a float |
| `Airport_fee` | `airport_fee` | decimal(10,2) | snake_case |
| *(new)* | `pickup_date` | date | from the pickup timestamp |
| *(new)* | `pickup_hour` | int | 0–23 |
| *(new)* | `trip_duration_min` | decimal(8,2) | (dropoff − pickup) in minutes |
| `source_month` | `source_month` | string | carried over |
| *(new)* | `silver_loaded_at` | timestamp | |

Columns not in this table (`source_file`, `ingested_at`) are **not** carried to Silver. The column order above is the order to use.

**Hints**
- `dropDuplicates(SRC_COLS)` removes exact duplicates; the difference of two counts tells you how many.
- `.cast("decimal(10,2)")` for money. An unknown payment type must become `Other`, never an error.
- `F.to_date`, `F.hour`, `F.unix_timestamp` cover the derived columns.
- Write with `overwrite` to `silver/taxi_trips/`, then `printSchema()` and count per `source_month`.

**Pause and observe →** Why decimal for money? Why set an out-of-range passenger count to `NULL` rather than reject the trip, when you *rejected* a ≤ 0 fare? Report §7 asks you to fill a before/after schema table.

In [ ]:
# Step 3: your code here

---
## Step 4 — Silver weather

Read `bronze.weather_daily` and write `silver/weather_daily/`:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `time` | `weather_date` | date | one row per date, drop duplicates on the date |
| `temperature_2m_max` | `temp_max_c` | decimal(5,1) | |
| `temperature_2m_min` | `temp_min_c` | decimal(5,1) | |
| `precipitation_sum` | `precip_mm` | decimal(6,1) | |
| `snowfall_sum` | `snowfall_cm` | decimal(6,1) | |
| `wind_speed_10m_max` | `wind_max_kmh` | decimal(6,1) | |
| *(new)* | `is_rainy` | boolean | true when precipitation is **1.0 mm or more** |
| *(new)* | `silver_loaded_at` | timestamp | |

**Pause and observe →** Why encode the units (`_c`, `_mm`, `_cm`, `_kmh`) in the column names? Report §7.

In [ ]:
# Step 4: your code here

---
## Step 5 — Register, reconcile and sanity-check

1. Create schema `silver` and register the three **external** tables from the contract table.
2. Run the check cell below (given). Fix every ❌. It also reconciles **bronze = silver + rejected + duplicates removed**.
3. Write **one query** of your own that joins `silver.taxi_trips` to `silver.weather_daily` on the date and returns **trips per day for rainy vs dry days**. Put the result in report §9. This is only a sanity check that the two tables fit together, not an analysis.

> **Git checkpoint:** commit this notebook and your filled report (`git commit -m "notebook 2: silver layer + report"`).

In [ ]:
# Step 5: register the three external tables here

In [ ]:
# Step 5: your rainy-vs-dry join query here

In [ ]:
from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

EXPECTED_SILVER_TAXI = {
    "vendor_id": "int", "pickup_ts": "timestamp", "dropoff_ts": "timestamp", "passenger_count": "int",
    "trip_distance_miles": "decimal(8,2)", "rate_code_id": "int", "store_and_fwd_flag": "boolean",
    "pickup_location_id": "int", "dropoff_location_id": "int", "payment_type_id": "int",
    "payment_type_desc": "string", "fare_amount": "decimal(10,2)", "extra": "decimal(10,2)",
    "mta_tax": "decimal(10,2)", "tip_amount": "decimal(10,2)", "tolls_amount": "decimal(10,2)",
    "improvement_surcharge": "decimal(10,2)", "total_amount": "decimal(10,2)",
    "congestion_surcharge": "decimal(10,2)", "airport_fee": "decimal(10,2)", "cbd_congestion_fee": "decimal(10,2)",
    "pickup_date": "date", "pickup_hour": "int", "trip_duration_min": "decimal(8,2)",
    "source_month": "string", "silver_loaded_at": "timestamp",
}
EXPECTED_SILVER_WEATHER = {
    "weather_date": "date", "temp_max_c": "decimal(5,1)", "temp_min_c": "decimal(5,1)",
    "precip_mm": "decimal(6,1)", "snowfall_cm": "decimal(6,1)", "wind_max_kmh": "decimal(6,1)",
    "is_rainy": "boolean", "silver_loaded_at": "timestamp",
}
ALLOWED_REASONS = {"pickup_outside_source_month", "dropoff_not_after_pickup", "duration_over_6h",
                   "distance_invalid", "fare_invalid"}

# ---------------- external tables in the right place ----------------
for t in ["taxi_trips", "taxi_trips_rejected", "weather_daily"]:
    info = table_info(f"{CATALOG}.silver.{t}")
    check(f"silver.{t} is EXTERNAL at <container>/silver/{t}",
          info.get("Type") == "EXTERNAL" and info.get("Location", "").rstrip("/").endswith(f"/silver/{t}"))

# ---------------- silver taxi: contract ----------------
st = spark.table(f"{CATALOG}.silver.taxi_trips")
check("silver.taxi_trips has exactly the agreed columns and types", dict(st.dtypes) == EXPECTED_SILVER_TAXI)
stats = st.agg(
    F.min("trip_duration_min").alias("min_dur"), F.max("trip_duration_min").alias("max_dur"),
    F.min("fare_amount").alias("min_fare"), F.max("fare_amount").alias("max_fare"),
    F.min("trip_distance_miles").alias("min_dist"), F.max("trip_distance_miles").alias("max_dist"),
    F.min("passenger_count").alias("min_pax"), F.max("passenger_count").alias("max_pax"),
    F.sum((F.date_format("pickup_ts", "yyyy-MM") != F.col("source_month")).cast("int")).alias("wrong_month"),
).first()
check("no trip with duration <= 0 or > 360 min", stats.min_dur > 0 and stats.max_dur <= 360)
check("no fare <= 0 or >= 500", stats.min_fare > 0 and stats.max_fare < 500)
check("no distance <= 0 or >= 100", stats.min_dist > 0 and stats.max_dist < 100)
check("passenger_count is NULL or between 1 and 6", stats.min_pax >= 1 and stats.max_pax <= 6)
check("every pickup falls in its own source_month", stats.wrong_month == 0)

# ---------------- rejected table ----------------
rj = spark.table(f"{CATALOG}.silver.taxi_trips_rejected")
check("rejected table has reject_reason and rejected_at", {"reject_reason", "rejected_at"} <= set(rj.columns))
reasons = {r.reject_reason for r in rj.select("reject_reason").distinct().collect()}
check("every reject_reason is one of the 5 agreed strings", reasons <= ALLOWED_REASONS)

# ---------------- reconciliation: nothing disappears silently ----------------
b_n, s_n, r_n = spark.table(f"{CATALOG}.bronze.taxi_trips").count(), st.count(), rj.count()
check(f"bronze ({b_n:,}) == silver ({s_n:,}) + rejected ({r_n:,}) + duplicates removed ({DUPLICATES_REMOVED:,})",
      b_n == s_n + r_n + DUPLICATES_REMOVED)

# ---------------- silver weather ----------------
sw = spark.table(f"{CATALOG}.silver.weather_daily")
check("silver.weather_daily has the agreed columns and types", dict(sw.dtypes) == EXPECTED_SILVER_WEATHER)
check("weather: 92 rows, one per date", sw.count() == 92 and sw.select("weather_date").distinct().count() == 92)

summary()
print(f"Bronze {b_n:,} | Silver {s_n:,} | Rejected {r_n:,} | Duplicates removed {DUPLICATES_REMOVED:,}")